# Credit Risk Probability Scoring using ANN

This notebook reproduces the modular training and evaluation workflow used by the GitHub project. The default data is deterministic and synthetic, so no customer information is required.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data_preprocessing import load_dataset, split_dataset, fit_transform_splits
from src.model_training import build_credit_risk_ann, set_reproducibility
from src.model_evaluation import compute_metrics, tune_threshold, save_evaluation_plots
from src.risk_scoring import add_risk_outputs

## 1. Load the synthetic applicant dataset

In [ ]:
data = load_dataset()
print(data.shape)
data.head()

In [ ]:
data["default_flag"].value_counts(normalize=True).rename("share")

## 2. Stratified train / validation / test split and preprocessing

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = split_dataset(data)
preprocessor, X_train_p, X_val_p, X_test_p = fit_transform_splits(X_train, X_val, X_test)
print(X_train.shape, X_val.shape, X_test.shape)
print("Processed input dimension:", X_train_p.shape[1])

## 3. Build and train the ANN

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import callbacks

set_reproducibility()
classes = np.unique(y_train)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weight = {int(label): float(weight) for label, weight in zip(classes, weights)}

model = build_credit_risk_ann(X_train_p.shape[1])
model.summary()

In [ ]:
history = model.fit(
    X_train_p, np.asarray(y_train).astype("float32"),
    validation_data=(X_val_p, np.asarray(y_val).astype("float32")),
    epochs=40, batch_size=256, class_weight=class_weight,
    callbacks=[
        callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=8, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor="val_auc", mode="max", factor=0.5, patience=4),
    ],
    verbose=1,
)

## 4. Tune the decision threshold on validation F1

In [ ]:
val_prob = model.predict(X_val_p, verbose=0).ravel()
test_prob = model.predict(X_test_p, verbose=0).ravel()
best_threshold, threshold_table = tune_threshold(y_val, val_prob, objective="f1")
best_threshold, threshold_table.sort_values("f1", ascending=False).head(10)

## 5. Test metrics and probability scoring

In [ ]:
metrics = compute_metrics(y_test, test_prob, best_threshold)
pd.Series({k: v for k, v in metrics.items() if isinstance(v, (int, float))})

In [ ]:
scored = add_risk_outputs(X_test, test_prob, classification_threshold=best_threshold)
scored[["risk_probability", "risk_category", "decision_recommendation"]].head(10)

In [ ]:
scored["risk_category"].value_counts()

## 6. Save evaluation visuals and model artifacts

In [ ]:
from src.config import MODEL_DIR, OUTPUT_DIR
from src.data_preprocessing import save_preprocessor, export_portable_schema
from src.model_evaluation import save_metrics

MODEL_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
model.save(MODEL_DIR / "final_credit_risk_ann_model.keras")
save_preprocessor(preprocessor, MODEL_DIR / "preprocessor.joblib")
export_portable_schema(preprocessor, MODEL_DIR / "preprocessing_schema.json")
save_metrics(metrics, OUTPUT_DIR / "model_metrics.json")
save_evaluation_plots(y_test, test_prob, best_threshold, OUTPUT_DIR)

## Interpretation

The classification threshold and operational risk bands solve different problems. Validation F1 selects a label cutoff, while risk bands support prioritization. In production, both would be governed by expected loss, approval capacity, calibration, fairness, regulation, and monitoring.